In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Kaggle notebook code (GPU recommended)
# ------------------------------------

import json, os, random 
from collections import Counter, defaultdict

import torch
from torch.utils.data import DataLoader

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    # Trainer,
    DataCollatorWithPadding,
    set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = "/kaggle/input/data4good-updaedcontextsplit/train_80.json"  # provided path in this environment; on Kaggle set accordingly

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2026-02-01 17:09:59.283503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769965799.505671      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769965799.569219      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769965800.113127      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769965800.113187      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769965800.113190      55 computation_placer.cc:177] computation placer alr

/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv
/kaggle/input/data4good-updaedcontextsplit/val_10.json
/kaggle/input/data4good-updaedcontextsplit/train_80.json
/kaggle/input/data4good-updaedcontextsplit/test_10.json
/kaggle/input/data4good-original-data-split/val_10.json
/kaggle/input/data4good-original-data-split/train_80.json
/kaggle/input/data4good-original-data-split/test_10.json
/kaggle/input/gpt-labelled-dataset/test10-gpt-labelled.csv


In [2]:
CLASS_NAMES = ["factual", "contradiction", "irrelevant"]

def build_premise(ex):
    # You can tweak this, but keep it consistent across runs
    ctx = (ex.get("context") or "").strip()
    q   = (ex.get("question") or "").strip()
    if ctx:
        return f"{ctx}\n\nQuestion: {q}"
    return f"Question: {q}"

def build_hypothesis(ex):
    return (ex.get("answer") or "").strip()

def get_mnli_label_ids(model_config):
    """
    Returns dict: {"entailment": id, "contradiction": id, "neutral": id}
    Works across models where config.label2id may be inconsistent/cased.
    """
    l2i = model_config.label2id or {}
    # normalize
    norm = {str(k).lower(): int(v) for k, v in l2i.items()}
    out = {}

    # common names in MNLI configs
    for key in ["entailment", "contradiction", "neutral"]:
        if key in norm:
            out[key] = norm[key]

    # sometimes keys are like "LABEL_0"/"LABEL_1"/"LABEL_2"
    # and config.id2label provides meaning
    if len(out) < 3 and getattr(model_config, "id2label", None):
        i2l = {int(k): str(v).lower() for k, v in model_config.id2label.items()}
        for i, name in i2l.items():
            if "entail" in name:
                out["entailment"] = i
            elif "contra" in name:
                out["contradiction"] = i
            elif "neutral" in name:
                out["neutral"] = i

    # final fallback (MNLI default often: 0=contradiction,1=neutral,2=entailment)
    if len(out) < 3:
        out = {"contradiction": 0, "neutral": 1, "entailment": 2}

    return out

# dataset label mapping:
# factual -> entailment, contradiction -> contradiction, irrelevant -> neutral
def to_mnli_target_id(ex, mnli_ids):
    t = ex["type"]
    if t == "factual":
        return mnli_ids["entailment"]
    if t == "contradiction":
        return mnli_ids["contradiction"]
    if t == "irrelevant":
        return mnli_ids["neutral"]
    raise ValueError(f"Unknown type: {t}")

def preds_to_task_label(pred_mnli_id, mnli_ids):
    # reverse mapping MNLI decision -> task class
    if pred_mnli_id == mnli_ids["entailment"]:
        return "factual"
    if pred_mnli_id == mnli_ids["contradiction"]:
        return "contradiction"
    if pred_mnli_id == mnli_ids["neutral"]:
        return "irrelevant"
    # if model outputs something unexpected, call it irrelevant
    return "irrelevant"

def compute_metrics_task(y_true, y_pred, title=""):
    # overall
    overall_acc = float(np.mean([a == b for a, b in zip(y_true, y_pred)]))
    overall_f1_macro = float(f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro"))

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in CLASS_NAMES:
        idxs = [i for i, yt in enumerate(y_true) if yt == c]
        if len(idxs) == 0:
            per_class_acc[c] = None
        else:
            per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs]))

    report = classification_report(
        y_true, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0
    )

    # print clean summary
    print("\n" + "="*80)
    if title:
        print(title)
    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Overall F1 (macro): {overall_f1_macro:.4f}")
    print("\nPer-class Accuracy:")
    for c in CLASS_NAMES:
        v = per_class_acc[c]
        print(f"  {c:14s} {('NA' if v is None else f'{v:.4f}')}")
    print("\nPer-class F1:")
    for c in CLASS_NAMES:
        print(f"  {c:14s} {report[c]['f1-score']:.4f}")

    return {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1_macro,
        "per_class_accuracy": per_class_acc,
        "per_class_f1": {c: float(report[c]["f1-score"]) for c in CLASS_NAMES},
    }

In [ ]:
import os, json, gc, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
gc.collect(); torch.cuda.empty_cache()

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # use public model for reproducibility
# MODEL_NAME = "microsoft/deberta-v3-base-mnli"  # if you want faster + less OOM risk

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

MAX_LENGTH = 250
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

SPLIT_DIR = "./splits_80_10_10"     # where you saved train_80.json etc.
OUT_DIR = "./outputs_80_10_10"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# Load split json files
# -----------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json('/kaggle/input/data4good-updaedcontextsplit/train_80.json')
val_data   = load_json('/kaggle/input/data4good-updaedcontextsplit/val_10.json')
test_data  = load_json('/kaggle/input/data4good-updaedcontextsplit/test_10.json')

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist:", train_df["type"].value_counts().to_dict())
print("Val dist:", val_df["type"].value_counts().to_dict())
print("Test dist:", test_df["type"].value_counts().to_dict())

# -----------------------
# Tokenize
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Model
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# memory saver
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# -----------------------
# Metrics (paper required)
# -----------------------
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = metrics_from_preds(labels.tolist(), preds.tolist())
    # Trainer expects flat numeric values
    return {"accuracy": m["overall_accuracy"], "f1_macro": m["overall_f1_macro"]}

# -----------------------
# TrainingArguments (handles eval_strategy API mismatch)
# -----------------------
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

args_kwargs = dict(
    output_dir=os.path.join(OUT_DIR, "ft_model"),
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)
args_kwargs[eval_key] = "epoch"
args_kwargs = {k:v for k,v in args_kwargs.items() if k in allowed}

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_trainer,
)

gc.collect(); torch.cuda.empty_cache()
trainer.train()

# -----------------------
# Evaluate on VAL + TEST, save metrics + predictions
# -----------------------
def predict_and_save(split_name, df, ds):
    pred = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    conf = probs.max(axis=-1)
    true_ids = df["label"].values

    # metrics
    met = metrics_from_preds(true_ids.tolist(), pred_ids.tolist())
    met_row = {"model": MODEL_NAME, "split": split_name, **met}

    # predictions csv
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = [id2label[i] for i in pred_ids]
    out["pred_conf"] = conf
    for i,lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}.csv")
    out.to_csv(pred_path, index=False)

    return met_row, pred_path

val_metrics_row, val_pred_path = predict_and_save("val", val_df, val_ds)
test_metrics_row, test_pred_path = predict_and_save("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_metrics_row, test_metrics_row])
metrics_path = os.path.join(OUT_DIR, "finetune_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


Sizes: 16816 2102 2103
Train dist: {'factual': 13944, 'contradiction': 1454, 'irrelevant': 1418}
Val dist: {'factual': 1743, 'contradiction': 182, 'irrelevant': 177}
Test dist: {'factual': 1744, 'contradiction': 182, 'irrelevant': 177}


Map:   0%|          | 0/16816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2102 [00:00<?, ? examples/s]

Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


In [1]:
import os, gc, torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

gc.collect()
torch.cuda.empty_cache()

# Path where Trainer saved your best checkpoint
SAVED_DIR = os.path.join('/kaggle/working/outputs_80_10_10', "ft_model/checkpoint-1051")  # same as output_dir used in training

# Load tokenizer + fine-tuned model from disk
tokenizer = AutoTokenizer.from_pretrained(SAVED_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(SAVED_DIR)

# If you want to enforce label names (optional, but helps readability)
model.config.id2label = id2label
model.config.label2id = label2id

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

2026-01-29 14:43:22.517838: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769697802.768287      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769697802.842666      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769697803.477807      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697803.477853      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697803.477856      55 computation_placer.cc:177] computation placer alr

In [2]:

MAX_LENGTH = 250
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# Minimal args just for prediction (eval batch size matters)
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

pred_args_kwargs = dict(
    output_dir=os.path.join('/kaggle/working/outputs_80_10_10', "serapi_pred_only"),
    per_device_eval_batch_size=EVAL_BS,
    fp16=True,
    report_to="none",
)

pred_args_kwargs[eval_key] = "no"
pred_args_kwargs = {k:v for k,v in pred_args_kwargs.items() if k in allowed}
pred_args = TrainingArguments(**pred_args_kwargs)

pred_trainer = Trainer(
    model=model,
    args=pred_args,
    data_collator=data_collator,
)

LABELS = ["factual", "contradiction", "irrelevant"]



In [7]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json('/kaggle/input/data4good-updaedcontextsplit/train_80.json')
val_data   = load_json('/kaggle/input/data4good-updaedcontextsplit/val_10.json')
test_data  = load_json('/kaggle/input/data4good-updaedcontextsplit/test_10.json')

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

In [8]:


def metrics_table(y_true_str, y_pred_str):
    # overall
    overall_acc = float(np.mean(np.array(y_true_str) == np.array(y_pred_str)))
    overall_f1  = float(f1_score(y_true_str, y_pred_str, labels=LABELS, average="macro", zero_division=0))

    # per-class accuracy
    per_class_acc = {}
    for c in LABELS:
        idxs = np.where(np.array(y_true_str) == c)[0]
        per_class_acc[c] = float(np.mean(np.array(y_pred_str)[idxs] == c)) if len(idxs) else None

    # per-class f1
    rep = classification_report(
        y_true_str, y_pred_str, labels=LABELS, output_dict=True, zero_division=0
    )

    row = {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1,
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(rep["factual"]["f1-score"]),
        "f1_contradiction": float(rep["contradiction"]["f1-score"]),
        "f1_irrelevant": float(rep["irrelevant"]["f1-score"]),
    }
    return row

@torch.no_grad()
def predict_split(split_name, df, ds):
    pred = pred_trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    pred_labels = [id2label[int(i)] for i in pred_ids]

    true_labels = df["type"].astype(str).str.lower().tolist()

    # metrics
    met = metrics_table(true_labels, pred_labels)
    met_row = {"model": SAVED_DIR, "split": split_name, **met}

    # save predictions
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = pred_labels
    out["pred_conf"] = probs.max(axis=-1)
    for i, lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}_LOADED.csv")
    out.to_csv(pred_path, index=False)

    print(split_name, met)
    return met_row, pred_path

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

Map:   0%|          | 0/16816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2102 [00:00<?, ? examples/s]

Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

In [ ]:
import os, json
from pathlib import Path
from collections import Counter
OUT_DIR = Path("./splits_80_10_10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Predict + metrics on VAL and TEST using LOADED model
val_row, val_pred_path = predict_split("val", val_df, val_ds)
test_row, test_pred_path = predict_split("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_row, test_row])
metrics_path = os.path.join('/kaggle/working/outputs_80_10_10', "serapi_added_finetune_metrics_LOADED.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [3]:
test_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_test_LOADED.csv")
val_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv")

In [10]:
from sklearn.metrics import accuracy_score, f1_score
LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

def metrics_by_initial_context(test_data, test_df_pred):
    assert len(test_data) == len(test_df_pred)

    results = {}

    # initially NO context (in original test_data)
    idx_no_ctx = test_data["context"].isna() | test_data["context"].eq("")
    df0 = test_df_pred.loc[idx_no_ctx]

    results["initially_no_context"] = metrics_from_preds(
        df0.label.tolist(),
        df0.pred_id.tolist()
    )
    print("initially_no_context:", df0.shape)

    # initially HAS context
    idx_ctx = ~idx_no_ctx
    df1 = test_df_pred.loc[idx_ctx]

    results["initially_has_context"] = metrics_from_preds(
        df1.label.tolist(),
        df1.pred_id.tolist()
    )
    print("initially_has_context:", df1.shape)

    return results


In [6]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

val_data   = load_json('/kaggle/input/data4good-original-data-split/val_10.json')
test_data  = load_json('/kaggle/input/data4good-original-data-split/test_10.json')

In [15]:
results = metrics_by_initial_context(pd.DataFrame(test_data), test_df_pred)

initially_no_context: (183, 13)
initially_has_context: (1920, 13)


In [16]:
pd.DataFrame([results['initially_no_context']])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.989071,0.978155,0.986667,1.0,1.0,0.993289,0.941176,1.0


In [17]:
pd.DataFrame([results['initially_has_context']])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.99375,0.98649,0.998118,0.951807,0.99375,0.996243,0.966361,0.996865


## Disagreements between GPT and deberta

In [13]:
import pandas as pd
import numpy as np
gpt_df = pd.read_csv("/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv")
test_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv")

In [5]:
test_df_pred

,answer,type,question,context,premise_text,hypothesis_text,label,pred_id,pred_label,pred_conf,prob_factual,prob_contradiction,prob_irrelevant
0,"In Popper's view, we don't need to search for ...",factual,What don't we need to look for about theories ...,"To Popper, who was an anti-justificationist, t...","To Popper, who was an anti-justificationist, t...","In Popper's view, we don't need to search for ...",0,0,factual,0.999918,0.999918,0.000078,0.000004
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,When was the phrase Irano-Aryan first used?,"The term ""Irano-Aryan"" (and its related forms ...","The term ""Irano-Aryan"" (and its related forms ...","The term ""Irano-Aryan"" was first used in 1878.",0,0,factual,0.985646,0.985646,0.014272,0.000082
2,The Augmented Foundation Programme is required...,factual,Where is the Augmented Programme required for ...,Fetuvalu offers the Cambridge syllabus. Motufo...,Fetuvalu offers the Cambridge syllabus. Motufo...,The Augmented Foundation Programme is required...,0,0,factual,0.999867,0.999867,0.000128,0.000005
3,The general election for Tucson's city council...,factual,When is Tucson's city council general election?,Both the council members and the mayor serve f...,Both the council members and the mayor serve f...,The general election for Tucson's city council...,0,0,factual,0.999894,0.999894,0.000100,0.000006
4,The organization in Plymouth named after Sir A...,factual,What Plymouth organization is named for Sir Al...,Plymouth is home to the Marine Biological Asso...,Plymouth is home to the Marine Biological Asso...,The organization in Plymouth named after Sir A...,0,0,factual,0.999905,0.999905,0.000091,0.000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,What is the head of each Municipio called?,"As of 2010[update], the city of Montevideo has...","As of 2010[update], the city of Montevideo has...",The head of each Municipio is called an alcald...,0,0,factual,0.999925,0.999925,0.000071,0.000004
2098,Alan Dershowitz characterized Sudan as a gover...,factual,How did Alan Dershozitz describe the Sudan?,Alan Dershowitz described Sudan as an example ...,Alan Dershowitz described Sudan as an example ...,Alan Dershowitz characterized Sudan as a gover...,0,0,factual,0.999941,0.999941,0.000053,0.000005
2099,The name of the tower that was the first purpo...,factual,What's the name of the wall tower that was the...,The name of the wall tower that was the first ...,The name of the wall tower that was the first ...,The name of the tower that was the first purpo...,0,0,factual,0.999096,0.999096,0.000889,0.000015
2100,"Thomas Young first used the term ""energy"" inst...",factual,"When did Thomas Young use the term ""energy"" in...","In 1807, Thomas Young was possibly the first t...","In 1807, Thomas Young was possibly the first t...","Thomas Young first used the term ""energy"" inst...",0,0,factual,0.999869,0.999869,0.000127,0.000004


In [3]:
test_df_pred[['answer', 'type', 'question', 'context']]

,answer,type,question,context
0,"In Popper's view, we don't need to search for ...",factual,What don't we need to look for about theories ...,"To Popper, who was an anti-justificationist, t..."
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,When was the phrase Irano-Aryan first used?,"The term ""Irano-Aryan"" (and its related forms ..."
2,The Augmented Foundation Programme is required...,factual,Where is the Augmented Programme required for ...,Fetuvalu offers the Cambridge syllabus. Motufo...
3,The general election for Tucson's city council...,factual,When is Tucson's city council general election?,Both the council members and the mayor serve f...
4,The organization in Plymouth named after Sir A...,factual,What Plymouth organization is named for Sir Al...,Plymouth is home to the Marine Biological Asso...
...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,What is the head of each Municipio called?,"As of 2010[update], the city of Montevideo has..."
2098,Alan Dershowitz characterized Sudan as a gover...,factual,How did Alan Dershozitz describe the Sudan?,Alan Dershowitz described Sudan as an example ...
2099,The name of the tower that was the first purpo...,factual,What's the name of the wall tower that was the...,The name of the wall tower that was the first ...
2100,"Thomas Young first used the term ""energy"" inst...",factual,"When did Thomas Young use the term ""energy"" in...","In 1807, Thomas Young was possibly the first t..."


In [4]:
gpt_df

,answer,type,context,question,model_label
0,"In Popper's view, we don't need to search for ...",factual,"To Popper, who was an anti-justificationist, t...",What don't we need to look for about theories ...,factual
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,NaN,When was the phrase Irano-Aryan first used?,contradiction
2,The Augmented Foundation Programme is required...,factual,Fetuvalu offers the Cambridge syllabus. Motufo...,Where is the Augmented Programme required for ...,factual
3,The general election for Tucson's city council...,factual,Both the council members and the mayor serve f...,When is Tucson's city council general election?,factual
4,The organization in Plymouth named after Sir A...,factual,Plymouth is home to the Marine Biological Asso...,What Plymouth organization is named for Sir Al...,factual
...,...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,"As of 2010[update], the city of Montevideo has...",What is the head of each Municipio called?,factual
2098,Alan Dershowitz characterized Sudan as a gover...,factual,Alan Dershowitz described Sudan as an example ...,How did Alan Dershozitz describe the Sudan?,factual
2099,The name of the tower that was the first purpo...,factual,NaN,What's the name of the wall tower that was the...,factual
2100,"Thomas Young first used the term ""energy"" inst...",factual,"In 1807, Thomas Young was possibly the first t...","When did Thomas Young use the term ""energy"" in...",factual


In [11]:
from sklearn.metrics import accuracy_score, classification_report
def metrics_from_preds(y_true, y_pred):
    # y_true = [id2label[i] for i in y_true_ids]
    # y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

In [14]:
pd.DataFrame([metrics_from_preds(gpt_df.type.tolist(), gpt_df.model_label.tolist())])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.972883,0.949824,0.971888,0.972527,0.983051,0.984598,0.876238,0.988636


In [15]:
import pandas as pd
import numpy as np

# -----------------------------
# 0) Paths (your uploaded files)
# -----------------------------
VAL_NLI_PATH  = "/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv"
TEST_NLI_PATH = "/kaggle/working/splits_80_10_10/predictions_test_LOADED.csv"
VAL_GPT_PATH  = "/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv"
TEST_GPT_PATH = "/kaggle/input/gpt-labelled-dataset/test10-gpt-labelled.csv"

OUT_CSV  = "/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv"
OUT_TEX  = "/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.tex"

# -----------------------------
# 1) Helpers (robust to quirks)
# -----------------------------
VALID = {"factual", "contradiction", "irrelevant"}
NUM2LBL = {0: "factual", 1: "contradiction", 2: "irrelevant"}

def normalize_label(x):
    """Map labels to {factual, contradiction, irrelevant}; anything else -> NaN."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return NUM2LBL.get(int(x), np.nan)
    s = str(x).strip().lower()
    s = s.replace(" ", "_")
    return s if s in VALID else np.nan

def context_group(x):
    """Split by whether context is present vs absent."""
    if pd.isna(x):
        return "absent"
    if isinstance(x, str) and x.strip() == "":
        return "absent"
    # if your pipeline uses placeholders, add them here:
    if isinstance(x, str) and x.strip().lower() in {"none", "null", "na", "n/a", "no_context"}:
        return "absent"
    return "present"

def pair_key(a, b):
    """Unordered label-pair key for disagreements (Factual↔Contradiction etc.)."""
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return "↔".join(sorted([a, b]))

def merge_preds(nli_df, gpt_df):
    """
    Merge NLI preds with GPT preds using stable text keys.
    Assumes columns exist:
      - NLI: question, answer, type, context, pred_label (or pred_id/label variants)
      - GPT: question, answer, type, context, model_label
    """
    nli = nli_df.copy()
    gpt = gpt_df.copy()

    # Normalize labels
    # NLI file already has pred_label strings; fallback to pred_id if needed
    if "pred_label" in nli.columns:
        nli["mnli"] = nli["pred_label"].apply(normalize_label)
    elif "pred_id" in nli.columns:
        nli["mnli"] = nli["pred_id"].apply(normalize_label)
    else:
        raise ValueError("NLI file missing pred_label/pred_id.")

    if "model_label" not in gpt.columns:
        raise ValueError("GPT file missing model_label.")
    gpt["gpt"] = gpt["model_label"].apply(normalize_label)

    # Merge on the most stable shared keys
    key_cols = [c for c in ["question", "answer", "type", "context"] if c in nli.columns and c in gpt.columns]
    if len(key_cols) < 2:
        raise ValueError(f"Too few merge keys found. Found keys={key_cols}")

    merged = nli.merge(
        gpt[key_cols + ["gpt"]],
        on=key_cols,
        how="left",
        validate="one_to_one",
    )

    merged["context_split"] = merged["context"].apply(context_group)

    # Validity + disagreement flags
    merged["mnli_valid"] = merged["mnli"].isin(list(VALID))
    merged["gpt_valid"]  = merged["gpt"].isin(list(VALID))
    merged["both_valid"] = merged["mnli_valid"] & merged["gpt_valid"]

    merged["agree"]    = merged["both_valid"] & (merged["mnli"] == merged["gpt"])
    merged["disagree"] = merged["both_valid"] & (merged["mnli"] != merged["gpt"])

    merged["pair"] = np.where(
        merged["disagree"],
        merged.apply(lambda r: pair_key(r["mnli"], r["gpt"]), axis=1),
        np.nan
    )

    return merged

def disagreement_summary_table(merged, dataset_name):
    """
    One table with:
      - # examples
      - # comparable (both valid)
      - # GPT missing/invalid (captures "ERROR..." etc.)
      - # disagreements and % among comparable
      - label-pair breakdown counts and % among disagreements
    """
    rows = []
    for split, g in merged.groupby("context_split", dropna=False):
        n = len(g)
        comparable = int(g["both_valid"].sum())
        gpt_bad = int((~g["gpt_valid"]).sum())  # includes NaN + non-label strings
        dis = int(g["disagree"].sum())

        dis_rate = (dis / comparable * 100.0) if comparable > 0 else np.nan

        pair_counts = g.loc[g["disagree"], "pair"].value_counts()

        def pc(k): return int(pair_counts.get(k, 0))
        # NOTE: keys are sorted strings ("contradiction↔factual", etc.)
        fc = pc("contradiction↔factual")
        fi = pc("factual↔irrelevant")
        ci = pc("contradiction↔irrelevant")

        rows.append({
            "dataset": dataset_name,
            "context": split,
            "n": n,
            "comparable_(both_valid)": comparable,
            "gpt_missing_or_invalid": gpt_bad,
            "disagree_n": dis,
            "disagree_%": round(dis_rate, 3) if comparable > 0 else np.nan,

            "F↔C_n": fc,
            "F↔I_n": fi,
            "C↔I_n": ci,

            "F↔C_%_of_dis": round((fc / dis * 100.0), 1) if dis > 0 else np.nan,
            "F↔I_%_of_dis": round((fi / dis * 100.0), 1) if dis > 0 else np.nan,
            "C↔I_%_of_dis": round((ci / dis * 100.0), 1) if dis > 0 else np.nan,
        })

    out = pd.DataFrame(rows)

    # Drop splits with zero comparable examples (avoids junk rows when "absent" is tiny)
    out = out[out["comparable_(both_valid)"] > 0].reset_index(drop=True)
    return out

def one_sentence_story(tab):
    """
    Generates a single paper-friendly sentence from the table.
    (You can paste this sentence under the table.)
    """
    # Aggregate across contexts per dataset
    sents = []
    for ds, g in tab.groupby("dataset"):
        comparable = g["comparable_(both_valid)"].sum()
        dis = g["disagree_n"].sum()
        rate = (dis / comparable * 100.0) if comparable > 0 else np.nan

        fc = g["F↔C_n"].sum()
        fi = g["F↔I_n"].sum()
        ci = g["C↔I_n"].sum()

        top_pair = max([("Factual↔Contradiction", fc), ("Factual↔Irrelevant", fi), ("Contradiction↔Irrelevant", ci)], key=lambda x: x[1])
        top_share = (top_pair[1] / dis * 100.0) if dis > 0 else np.nan

        sents.append((ds, rate, top_pair[0], top_share))

    # Prefer val+test in one sentence if both exist
    if set(tab["dataset"]) >= {"val", "test"}:
        v = [x for x in sents if x[0] == "val"][0]
        t = [x for x in sents if x[0] == "test"][0]
        return (
            f"MNLI and GPT disagree on {v[1]:.2f}% (val) and {t[1]:.2f}% (test) of comparable examples, "
            f"and most disagreements are {t[2]} flips (≈{t[3]:.1f}% of test disagreements), "
            f"showing remaining uncertainty concentrates on polarity rather than relevance."
        )

    # Fallback single-dataset sentence
    ds, rate, pair, share = sents[0]
    return (
        f"MNLI and GPT disagree on {rate:.2f}% of comparable examples ({ds}), "
        f"dominated by {pair} flips (≈{share:.1f}% of disagreements), "
        f"indicating uncertainty concentrates on polarity rather than relevance."
    )








In [16]:
# -----------------------------
# 2) Load + merge (val/test)
# -----------------------------
val_nli  = pd.read_csv(VAL_NLI_PATH)
test_nli = pd.read_csv(TEST_NLI_PATH)
val_gpt  = pd.read_csv(VAL_GPT_PATH)
test_gpt = pd.read_csv(TEST_GPT_PATH)

val_merged  = merge_preds(val_nli,  val_gpt)
test_merged = merge_preds(test_nli, test_gpt)


In [17]:
# -----------------------------
# 3) Build the ONE table
# -----------------------------
tab_val  = disagreement_summary_table(val_merged,  "val")
tab_test = disagreement_summary_table(test_merged, "test")
tab = pd.concat([tab_val, tab_test], ignore_index=True)

# Save for paper pipeline
tab.to_csv(OUT_CSV, index=False)

# LaTeX (booktabs-friendly)
latex = tab.to_latex(index=False, escape=False)
with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex)

print("Wrote:", OUT_CSV)
print("Wrote:", OUT_TEX)

Wrote: /kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv
Wrote: /kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.tex


In [18]:
# -----------------------------
# 4) The ONE sentence to paste
# -----------------------------
sentence = one_sentence_story(tab)
print("\nSentence for paper:\n", sentence)


Sentence for paper:
 MNLI and GPT disagree on 1.46% (val) and 1.42% (test) of comparable examples, and most disagreements are Factual↔Contradiction flips (≈96.3% of test disagreements), showing remaining uncertainty concentrates on polarity rather than relevance.


In [19]:

# -----------------------------
# 5) (Optional) sanity checks
# -----------------------------
# How many GPT rows were unusable (e.g., 'ERROR OCCURRED PLEASE FIX')?
print("\nGPT unusable labels:")
print("val :", int((~val_merged['gpt_valid']).sum()))
print("test:", int((~test_merged['gpt_valid']).sum()))

# Quick peek at the most common disagreement pairs overall
pairs = pd.concat([val_merged.loc[val_merged["disagree"], "pair"],
                   test_merged.loc[test_merged["disagree"], "pair"]])
print("\nTop disagreement pairs overall:")
print(pairs.value_counts().head(10))


GPT unusable labels:
val : 187
test: 195

Top disagreement pairs overall:
pair
contradiction↔factual    50
factual↔irrelevant        5
Name: count, dtype: int64


In [20]:
pd.read_csv("/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv")

,dataset,context,n,comparable_(both_valid),gpt_missing_or_invalid,disagree_n,disagree_%,F↔C_n,F↔I_n,C↔I_n,F↔C_%_of_dis,F↔I_%_of_dis,C↔I_%_of_dis
0,val,present,2102,1915,187,28,1.462,24,4,0,85.7,14.3,0.0
1,test,present,2102,1908,194,27,1.415,26,1,0,96.3,3.7,0.0


## Ensemble

In [3]:
import pandas as pd
import numpy as np

# =========================
# 0) Paths (your uploads)
# =========================
VAL_NLI_PATH  = "/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv"
VAL_GPT_PATH  = "/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv"

# If you also want test, uncomment and set paths:
TEST_NLI_PATH = "/kaggle/working/splits_80_10_10/predictions_test_LOADED.csv"
TEST_GPT_PATH = "/kaggle/input/gpt-labelled-dataset/test10-gpt-labelled.csv"

LABELS = ["factual", "contradiction", "irrelevant"]
VALID = set(LABELS)
NUM2LBL = {0: "factual", 1: "contradiction", 2: "irrelevant"}

# =========================
# 1) Robust helpers
# =========================
def normalize_label(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return NUM2LBL.get(int(x), np.nan)
    s = str(x).strip().lower().replace(" ", "_")
    return s if s in VALID else np.nan

def context_group(x):
    if pd.isna(x):
        return "absent"
    if isinstance(x, str) and x.strip() == "":
        return "absent"
    if isinstance(x, str) and x.strip().lower() in {"none", "null", "na", "n/a", "no_context"}:
        return "absent"
    return "present"

def guess_prob_columns(df):
    """
    Try to locate MNLI probability columns. Returns (prob_cols, mode) or (None, None).
    Supports:
      - p_factual/p_contradiction/p_irrelevant
      - prob_factual/prob_contradiction/prob_irrelevant
      - factual_prob/contradiction_prob/irrelevant_prob
    """
    candidates = [
        [f"p_{l}" for l in LABELS],
        [f"prob_{l}" for l in LABELS],
        [f"{l}_prob" for l in LABELS],
    ]
    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols, "cols"
    return None, None

def add_confidence(df, prob_cols):
    probs = df[prob_cols].to_numpy(dtype=float)
    # top1 and top2 without sorting full row
    top2 = np.partition(probs, -2, axis=1)[:, -2:]   # two largest (unordered)
    top1 = top2.max(axis=1)
    top2v = top2.min(axis=1)

    out = df.copy()
    out["mnli_conf"] = top1
    out["mnli_margin"] = top1 - top2v
    return out

def macro_f1(y_true, y_pred, labels=LABELS):
    f1s = []
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

def per_class_acc(y_true, y_pred, labels=LABELS):
    out = {}
    for lab in labels:
        m = (y_true == lab)
        out[f"acc_{lab}"] = float(np.mean(y_pred[m] == lab)) if np.any(m) else np.nan
    return out

def metrics(y_true, y_pred, labels=LABELS):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    overall_acc = float(np.mean(y_true == y_pred))
    out = {"overall_accuracy": overall_acc, "overall_f1_macro": macro_f1(y_true, y_pred, labels)}
    out.update(per_class_acc(y_true, y_pred, labels))

    # per-class f1 + balanced metrics
    f1s = []
    recalls = []
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        tn = np.sum((y_true != lab) & (y_pred != lab))  # not used but ok

        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        f1s.append(f1)
        recalls.append(rec)

        out[f"f1_{lab}"] = float(f1)

    out["balanced_acc"] = float(np.mean(recalls))
    out["balanced_f1"] = float(np.mean(f1s))
    return out

def merge_nli_gpt(nli_df, gpt_df):
    nli = nli_df.copy()
    gpt = gpt_df.copy()

    # Identify GT label column
    # Common: 'label' is numeric id; could also be string.
    if "label" not in nli.columns:
        raise ValueError("NLI file must contain ground-truth column 'label' for performance checks.")

    nli["label_norm"] = nli["label"].apply(normalize_label)

    # NLI prediction
    if "pred_label" in nli.columns:
        nli["mnli"] = nli["pred_label"].apply(normalize_label)
    elif "pred_id" in nli.columns:
        nli["mnli"] = nli["pred_id"].apply(normalize_label)
    else:
        raise ValueError("NLI file missing pred_label/pred_id.")

    # GPT prediction
    if "model_label" not in gpt.columns:
        raise ValueError("GPT file missing 'model_label'.")
    gpt["gpt"] = gpt["model_label"].apply(normalize_label)

    # Merge keys (use what's available; validate one-to-one)
    key_cols = [c for c in ["question", "answer", "type", "context"] if c in nli.columns and c in gpt.columns]
    if len(key_cols) < 2:
        raise ValueError(f"Too few merge keys found. Found keys={key_cols}. Need at least 2 shared keys.")

    merged = nli.merge(
        gpt[key_cols + ["gpt"]],
        on=key_cols,
        how="left",
        validate="one_to_one"
    )

    merged["context_split"] = merged["context"].apply(context_group) if "context" in merged.columns else "unknown"

    merged["mnli_valid"] = merged["mnli"].isin(LABELS)
    merged["gpt_valid"]  = merged["gpt"].isin(LABELS)
    merged["both_valid"] = merged["mnli_valid"] & merged["gpt_valid"]

    return merged


In [4]:

# =========================
# 2) Ensemble arbitration
# =========================
def ensemble_arbitrate(df, T_conf=0.90, T_margin=0.20, prefer="gpt_when_uncertain"):
    """
    Arbitration rule (best practical):
      - If GPT invalid -> use MNLI
      - Else if MNLI confident -> use MNLI
      - Else -> use GPT (only when MNLI is uncertain)
    Confidence uses MNLI probability columns if present. If not present, it falls back:
      - If MNLI and GPT disagree -> use GPT (if valid), else MNLI
      - If they agree -> use MNLI (or GPT; same)
    """
    out = df.copy()

    prob_cols, mode = guess_prob_columns(out)

    if prob_cols is not None:
        out = add_confidence(out, prob_cols)
        mnli_confident = (out["mnli_conf"] >= T_conf) | (out["mnli_margin"] >= T_margin)

        out["ensemble_pred"] = np.where(
            (~out["gpt_valid"]) | mnli_confident,
            out["mnli"],
            out["gpt"]
        )
        out["ensemble_mode"] = "confidence_arbitration"
        out["used_gpt"] = (out["ensemble_pred"] == out["gpt"]) & out["gpt_valid"] & (~mnli_confident)
    else:
        # Fallback (no MNLI probs): only meaningful place to arbitrate is disagreement rows.
        # This is weaker than confidence arbitration but still valid.
        disagree = out["both_valid"] & (out["mnli"] != out["gpt"])
        if prefer == "gpt_when_uncertain":
            out["ensemble_pred"] = np.where(disagree, out["gpt"], out["mnli"])
        else:
            out["ensemble_pred"] = out["mnli"]
        out["ensemble_mode"] = "disagreement_only_fallback"
        out["used_gpt"] = disagree & out["gpt_valid"]

    return out



In [5]:
# =========================
# 3) Run on validation + report
# =========================
val_nli = pd.read_csv(VAL_NLI_PATH)
val_gpt = pd.read_csv(VAL_GPT_PATH)

val_merged = merge_nli_gpt(val_nli, val_gpt)

# Build ensemble predictions (tune T_conf/T_margin on val if you have probs)
val_ens = ensemble_arbitrate(val_merged, T_conf=0.80, T_margin=0.15)

# Performance on full validation (comparable GT rows)
val_eval = val_ens.dropna(subset=["label_norm", "mnli"]).copy()
val_eval = val_eval[val_eval["label_norm"].isin(LABELS)]

mnli_metrics = metrics(val_eval["label_norm"].values, val_eval["mnli"].values)
gpt_metrics  = metrics(val_eval["label_norm"].values, val_eval["gpt"].fillna("irrelevant").values)  # GPT may be NaN; keep metrics stable
ens_metrics  = metrics(val_eval["label_norm"].values, val_eval["ensemble_pred"].values)

print("=== Validation metrics ===")
print("MNLI :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in mnli_metrics.items()})
print("GPT  :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in gpt_metrics.items()})
print("ENS  :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ens_metrics.items()})
print("Ensemble mode:", val_ens["ensemble_mode"].iloc[0])


=== Validation metrics ===
MNLI : {'overall_accuracy': 0.989534, 'overall_f1_macro': 0.97639, 'acc_factual': 0.996558, 'acc_contradiction': 0.923077, 'acc_irrelevant': 0.988701, 'f1_factual': 0.993991, 'f1_contradiction': 0.946479, 'f1_irrelevant': 0.988701, 'balanced_acc': 0.969445, 'balanced_f1': 0.97639}
GPT  : {'overall_accuracy': 0.911513, 'overall_f1_macro': 0.842112, 'acc_factual': 0.905909, 'acc_contradiction': 0.879121, 'acc_irrelevant': 1.0, 'f1_factual': 0.950632, 'f1_contradiction': 0.906516, 'f1_irrelevant': 0.669187, 'balanced_acc': 0.928343, 'balanced_f1': 0.842112}
ENS  : {'overall_accuracy': 0.990485, 'overall_f1_macro': 0.978547, 'acc_factual': 0.996558, 'acc_contradiction': 0.934066, 'acc_irrelevant': 0.988701, 'f1_factual': 0.994561, 'f1_contradiction': 0.952381, 'f1_irrelevant': 0.988701, 'balanced_acc': 0.973108, 'balanced_f1': 0.978547}
Ensemble mode: confidence_arbitration


In [29]:
pd.DataFrame([metrics(val_nli.type, val_nli.pred_label)])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant,balanced_acc,balanced_f1
0,0.989534,0.97639,0.996558,0.923077,0.988701,0.993991,0.946479,0.988701,0.969445,0.97639


In [30]:
pd.DataFrame([metrics(val_gpt.type, val_gpt.model_label)])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant,balanced_acc,balanced_f1
0,0.972883,0.949824,0.971888,0.972527,0.983051,0.984598,0.876238,0.988636,0.975822,0.949824


In [33]:

pd.DataFrame([metrics(val_ens.type, val_ens.ensemble_pred)])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant,balanced_acc,balanced_f1
0,0.990485,0.978547,0.996558,0.934066,0.988701,0.994561,0.952381,0.988701,0.973108,0.978547


In [4]:
# =========================
# 4) Resolution accuracy on disagreements (the key analysis)
# =========================
dis = val_eval[val_eval["both_valid"] & (val_eval["mnli"] != val_eval["gpt"])].copy()
print("\n=== Resolution on disagreements (validation) ===")
print("n_disagreements:", len(dis))
if len(dis) > 0:
    mnli_acc = (dis["mnli"] == dis["label_norm"]).mean() * 100
    gpt_acc  = (dis["gpt"] == dis["label_norm"]).mean() * 100
    ens_acc  = (dis["ensemble_pred"] == dis["label_norm"]).mean() * 100
    print(f"MNLI acc on disagreements: {mnli_acc:.2f}%")
    print(f"GPT  acc on disagreements: {gpt_acc:.2f}%")
    print(f"ENS  acc on disagreements: {ens_acc:.2f}%")



=== Resolution on disagreements (validation) ===
n_disagreements: 28
MNLI acc on disagreements: 42.86%
GPT  acc on disagreements: 57.14%
ENS  acc on disagreements: 50.00%


In [5]:
# =========================
# 5) Optional: how often ensemble actually uses GPT
# =========================
print("\n=== Ensemble usage stats (validation) ===")
print("GPT invalid count:", int((~val_ens["gpt_valid"]).sum()))
print("Ensemble used GPT (count):", int(val_ens["used_gpt"].sum()))
print("Ensemble used GPT (% of all):", round(val_ens["used_gpt"].mean() * 100, 3), "%")




=== Ensemble usage stats (validation) ===
GPT invalid count: 187
Ensemble used GPT (count): 2
Ensemble used GPT (% of all): 0.095 %


In [6]:
# =========================
# 6) (Optional) Tune thresholds on validation (only if prob cols exist)
# =========================
def tune_thresholds(val_df):
    prob_cols, _ = guess_prob_columns(val_df)
    if prob_cols is None:
        print("\nNo probability columns found. Skipping threshold tuning.")
        return None

    conf_grid = np.round(np.linspace(0.80, 0.98, 10), 2)
    margin_grid = np.round(np.linspace(0.05, 0.40, 8), 2)

    best = None
    best_score = -1.0

    eval_df = val_df.dropna(subset=["label_norm"])
    eval_df = eval_df[eval_df["label_norm"].isin(LABELS)]

    y_true = eval_df["label_norm"].to_numpy()

    for T_conf in conf_grid:
        for T_margin in margin_grid:
            tmp = ensemble_arbitrate(eval_df, T_conf=T_conf, T_margin=T_margin)
            score = macro_f1(y_true, tmp["ensemble_pred"].to_numpy())
            if score > best_score:
                best_score = score
                best = (T_conf, T_margin)

    print("\nBest (T_conf, T_margin):", best, "with val macro-F1:", round(best_score, 6))
    return best

# Uncomment to tune:
best_params = tune_thresholds(val_merged)
if best_params:
    val_ens_best = ensemble_arbitrate(val_merged, T_conf=best_params[0], T_margin=best_params[1])



Best (T_conf, T_margin): (np.float64(0.8), np.float64(0.15)) with val macro-F1: 0.978547


In [7]:
import numpy as np
import pandas as pd

LABELS = ["factual", "contradiction", "irrelevant"]

def guess_prob_columns(df):
    candidates = [
        [f"p_{l}" for l in LABELS],
        [f"prob_{l}" for l in LABELS],
        [f"{l}_prob" for l in LABELS],
    ]
    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols
    raise ValueError("Missing MNLI prob columns. Need p_factual/p_contradiction/p_irrelevant (or equivalent).")

def add_confidence(df, prob_cols):
    probs = df[prob_cols].to_numpy(dtype=float)
    top2 = np.partition(probs, -2, axis=1)[:, -2:]
    top1 = top2.max(axis=1)
    top2v = top2.min(axis=1)
    out = df.copy()
    out["mnli_conf"] = top1
    out["mnli_margin"] = top1 - top2v
    return out

def ensemble_disagreement_first(df, T_hi=0.97, M_hi=0.35):
    """
    Disagreement-first arbitration:
      - If GPT invalid -> MNLI
      - Else if MNLI==GPT -> MNLI
      - Else (disagree) -> GPT, unless MNLI is extremely confident -> MNLI
    """
    out = df.copy()
    prob_cols = guess_prob_columns(out)
    out = add_confidence(out, prob_cols)

    disagree = out["both_valid"] & (out["mnli"] != out["gpt"])
    mnli_extreme = (out["mnli_conf"] >= T_hi) | (out["mnli_margin"] >= M_hi)

    out["ensemble_pred"] = out["mnli"]  # default
    # if disagree and GPT valid, pick GPT
    out.loc[disagree & out["gpt_valid"], "ensemble_pred"] = out.loc[disagree & out["gpt_valid"], "gpt"]
    # but if MNLI extreme, override back to MNLI
    out.loc[disagree & out["gpt_valid"] & mnli_extreme, "ensemble_pred"] = out.loc[disagree & out["gpt_valid"] & mnli_extreme, "mnli"]

    out["used_gpt"] = disagree & out["gpt_valid"] & (~mnli_extreme)
    return out

def macro_f1(y_true, y_pred, labels=LABELS):
    f1s = []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

def resolution_acc(df):
    d = df[df["both_valid"] & (df["mnli"] != df["gpt"])].copy()
    if len(d) == 0:
        return np.nan
    return float(np.mean(d["ensemble_pred"] == d["label_norm"])) * 100.0

def tune_disagreement_first(val_df):
    # Tune for macro-F1 on full val, but report resolution accuracy too.
    T_grid = np.round(np.linspace(0.90, 0.995, 8), 3)   # extreme confidence
    M_grid = np.round(np.linspace(0.20, 0.55, 8), 2)    # extreme margin

    best = None
    best_f1 = -1.0
    best_res = None
    best_used = None

    eval_df = val_df.dropna(subset=["label_norm"]).copy()
    eval_df = eval_df[eval_df["label_norm"].isin(LABELS)]

    for T_hi in T_grid:
        for M_hi in M_grid:
            tmp = ensemble_disagreement_first(eval_df, T_hi=T_hi, M_hi=M_hi)
            f1 = macro_f1(tmp["label_norm"].to_numpy(), tmp["ensemble_pred"].to_numpy())
            if f1 > best_f1:
                best_f1 = f1
                best = (T_hi, M_hi)
                best_res = resolution_acc(tmp)
                best_used = float(tmp["used_gpt"].mean() * 100.0)

    print("Best (T_hi, M_hi):", best, "val macro-F1:", round(best_f1, 6),
          "| resolution acc on disagreements:", (None if best_res is None else round(best_res, 2)),
          "| used GPT %:", (None if best_used is None else round(best_used, 3)))
    return best

# ---- run tuning on validation ----
best_params = tune_disagreement_first(val_merged)

# ---- evaluate with best params ----
val_ens2 = ensemble_disagreement_first(val_merged, T_hi=best_params[0], M_hi=best_params[1])

# Full-val macro F1 (quick)
val_eval = val_ens2.dropna(subset=["label_norm"])
val_eval = val_eval[val_eval["label_norm"].isin(LABELS)]
print("Val macro-F1 (ENS2):", round(macro_f1(val_eval["label_norm"], val_eval["ensemble_pred"]), 6))

# Disagreement resolution stats
d = val_eval[val_eval["both_valid"] & (val_eval["mnli"] != val_eval["gpt"])]
print("n_disagreements:", len(d))
print("Resolution acc on disagreements (ENS2):", round(np.mean(d["ensemble_pred"] == d["label_norm"]) * 100, 2), "%")
print("ENS2 used GPT (count):", int(val_ens2["used_gpt"].sum()))
print("ENS2 used GPT (%):", round(val_ens2["used_gpt"].mean() * 100, 3), "%")


Best (T_hi, M_hi): (np.float64(0.9), np.float64(0.55)) val macro-F1: 0.979618 | resolution acc on disagreements: 53.57 | used GPT %: 0.143
Val macro-F1 (ENS2): 0.979618
n_disagreements: 28
Resolution acc on disagreements (ENS2): 53.57 %
ENS2 used GPT (count): 3
ENS2 used GPT (%): 0.143 %


In [8]:
import numpy as np
import pandas as pd

LABELS = ["factual", "contradiction", "irrelevant"]

def ensemble_disagree_choose_gpt(df):
    out = df.copy()

    # default MNLI
    out["ensemble_pred"] = out["mnli"]

    # if both valid and disagree, choose GPT
    mask = out["both_valid"] & (out["mnli"] != out["gpt"])
    out.loc[mask, "ensemble_pred"] = out.loc[mask, "gpt"]

    # if GPT invalid, it stays MNLI automatically
    out["used_gpt"] = mask
    out["ensemble_mode"] = "disagree_choose_gpt"
    return out

def macro_f1(y_true, y_pred, labels=LABELS):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1s = []
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))

def eval_all_and_disagreements(df, name="val"):
    # evaluate only where GT exists and valid
    eval_df = df.dropna(subset=["label_norm"]).copy()
    eval_df = eval_df[eval_df["label_norm"].isin(LABELS)]

    y = eval_df["label_norm"].to_numpy()

    # Full-set
    full = {
        "split": name,
        "n": len(eval_df),
        "mnli_acc": round(accuracy(y, eval_df["mnli"])*100, 3),
        "mnli_macro_f1": round(macro_f1(y, eval_df["mnli"])*100, 3),
        "gpt_acc": round(accuracy(y, eval_df["gpt"].fillna("irrelevant"))*100, 3),
        "gpt_macro_f1": round(macro_f1(y, eval_df["gpt"].fillna("irrelevant"))*100, 3),
        "ens_acc": round(accuracy(y, eval_df["ensemble_pred"])*100, 3),
        "ens_macro_f1": round(macro_f1(y, eval_df["ensemble_pred"])*100, 3),
        "ens_used_gpt_n": int(eval_df["used_gpt"].sum()),
        "ens_used_gpt_%": round(eval_df["used_gpt"].mean()*100, 3),
    }

    # Disagreement-only subset
    dis = eval_df[eval_df["both_valid"] & (eval_df["mnli"] != eval_df["gpt"])].copy()
    if len(dis) > 0:
        yd = dis["label_norm"].to_numpy()
        dis_stats = {
            "split": name,
            "n_disagree": len(dis),
            "mnli_acc_on_dis": round(accuracy(yd, dis["mnli"])*100, 3),
            "gpt_acc_on_dis": round(accuracy(yd, dis["gpt"])*100, 3),
            "ens_acc_on_dis": round(accuracy(yd, dis["ensemble_pred"])*100, 3),
        }
    else:
        dis_stats = {"split": name, "n_disagree": 0}

    return pd.DataFrame([full]), pd.DataFrame([dis_stats])

# ---- run on your merged validation dataframe ----
val_ens3 = ensemble_disagree_choose_gpt(val_merged)

full_tab, dis_tab = eval_all_and_disagreements(val_ens3, name="val")
print("=== Full-set performance (val) ===")
print(full_tab.to_string(index=False))

print("\n=== Disagreement resolution (val) ===")
print(dis_tab.to_string(index=False))


=== Full-set performance (val) ===
split    n  mnli_acc  mnli_macro_f1  gpt_acc  gpt_macro_f1  ens_acc  ens_macro_f1  ens_used_gpt_n  ens_used_gpt_%
  val 2102    98.953         97.639   91.151        84.211   99.144         98.13              28           1.332

=== Disagreement resolution (val) ===
split  n_disagree  mnli_acc_on_dis  gpt_acc_on_dis  ens_acc_on_dis
  val          28           42.857          57.143          57.143


In [9]:
import pandas as pd

LABELS = ["factual", "contradiction", "irrelevant"]

def disagreement_confusion_mnli_vs_gpt(df):
    """
    Confusion matrix of MNLI vs GPT on disagreement-only subset.
    Rows: MNLI, Columns: GPT
    """
    d = df[
        df["both_valid"] &
        (df["mnli"] != df["gpt"])
    ].copy()

    cm = pd.crosstab(
        d["mnli"],
        d["gpt"],
        rownames=["MNLI"],
        colnames=["GPT"],
        dropna=False
    )

    # ensure full label order
    cm = cm.reindex(index=LABELS, columns=LABELS, fill_value=0)
    return cm

cm_mnli_gpt = disagreement_confusion_mnli_vs_gpt(val_merged)
print("MNLI vs GPT disagreement confusion (val):")
print(cm_mnli_gpt)


MNLI vs GPT disagreement confusion (val):
GPT            factual  contradiction  irrelevant
MNLI                                             
factual              0             21           3
contradiction        3              0           0
irrelevant           1              0           0


In [10]:
def ensemble_confusion_on_disagreements(df):
    d = df[
        df["both_valid"] &
        (df["mnli"] != df["gpt"])
    ].copy()

    cm = pd.crosstab(
        d["label_norm"],
        d["ensemble_pred"],
        rownames=["Ground Truth"],
        colnames=["Ensemble"],
        dropna=False
    )

    cm = cm.reindex(index=LABELS, columns=LABELS, fill_value=0)
    return cm

cm_ens = ensemble_confusion_on_disagreements(val_ens)
print("\nGT vs Ensemble confusion on disagreements (val):")
print(cm_ens)



GT vs Ensemble confusion on disagreements (val):
Ensemble       factual  contradiction  irrelevant
Ground Truth                                     
factual             12              3           1
contradiction        8              2           0
irrelevant           2              0           0


## same analysis on test dataset

In [10]:
# =========================
# 3) Run on validation + report
# =========================
LABELS = ["factual", "contradiction", "irrelevant"]

def guess_prob_columns(df):
    candidates = [
        [f"p_{l}" for l in LABELS],
        [f"prob_{l}" for l in LABELS],
        [f"{l}_prob" for l in LABELS],
    ]
    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols, "cols"
    return None, None

test_nli = pd.read_csv(TEST_NLI_PATH)
test_gpt = pd.read_csv(TEST_GPT_PATH)

test_merged = merge_nli_gpt(test_nli, test_gpt)

# Build ensemble predictions (tune T_conf/T_margin on val if you have probs)
test_ens = ensemble_arbitrate(test_merged, T_conf=0.80, T_margin=0.05)

# Performance on full validation (comparable GT rows)
test_eval = test_ens.dropna(subset=["label_norm", "mnli"]).copy()
test_eval = test_eval[test_eval["label_norm"].isin(LABELS)]

mnli_metrics = metrics(test_eval["label_norm"].values, test_eval["mnli"].values)
gpt_metrics  = metrics(test_eval["label_norm"].values, test_eval["gpt"].fillna("irrelevant").values)  # GPT may be NaN; keep metrics stable
ens_metrics  = metrics(test_eval["label_norm"].values, test_eval["ensemble_pred"].values)

print("=== Validation metrics ===")
print("MNLI :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in mnli_metrics.items()})
print("GPT  :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in gpt_metrics.items()})
print("ENS  :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ens_metrics.items()})
print("Ensemble mode:", val_ens["ensemble_mode"].iloc[0])


=== Validation metrics ===
MNLI : {'overall_accuracy': 0.993343, 'overall_f1_macro': 0.985716, 'acc_factual': 0.997133, 'acc_contradiction': 0.956044, 'acc_irrelevant': 0.99435, 'f1_factual': 0.995991, 'f1_contradiction': 0.963989, 'f1_irrelevant': 0.997167, 'balanced_acc': 0.982509, 'balanced_f1': 0.985716}
GPT  : {'overall_accuracy': 0.908702, 'overall_f1_macro': 0.840268, 'acc_factual': 0.902523, 'acc_contradiction': 0.879121, 'acc_irrelevant': 1.0, 'f1_factual': 0.947622, 'f1_contradiction': 0.906516, 'f1_irrelevant': 0.666667, 'balanced_acc': 0.927215, 'balanced_f1': 0.840268}
ENS  : {'overall_accuracy': 0.993818, 'overall_f1_macro': 0.986765, 'acc_factual': 0.997133, 'acc_contradiction': 0.961538, 'acc_irrelevant': 0.99435, 'f1_factual': 0.996276, 'f1_contradiction': 0.966851, 'f1_irrelevant': 0.997167, 'balanced_acc': 0.984341, 'balanced_f1': 0.986765}
Ensemble mode: confidence_arbitration


In [14]:
pd.DataFrame([mnli_metrics, gpt_metrics, ens_metrics])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant,balanced_acc,balanced_f1
0,0.993343,0.985716,0.997133,0.956044,0.99435,0.995991,0.963989,0.997167,0.982509,0.985716
1,0.908702,0.840268,0.902523,0.879121,1.00000,0.947622,0.906516,0.666667,0.927215,0.840268
2,0.993818,0.986765,0.997133,0.961538,0.99435,0.996276,0.966851,0.997167,0.984341,0.986765


In [15]:
# =========================
# 6) (Optional) Tune thresholds on validation (only if prob cols exist)
# =========================
def tune_thresholds(val_df):
    prob_cols, _ = guess_prob_columns(val_df)
    if prob_cols is None:
        print("\nNo probability columns found. Skipping threshold tuning.")
        return None

    conf_grid = np.round(np.linspace(0.80, 0.98, 10), 2)
    margin_grid = np.round(np.linspace(0.05, 0.40, 8), 2)

    best = None
    best_score = -1.0

    eval_df = val_df.dropna(subset=["label_norm"])
    eval_df = eval_df[eval_df["label_norm"].isin(LABELS)]

    y_true = eval_df["label_norm"].to_numpy()

    for T_conf in conf_grid:
        for T_margin in margin_grid:
            tmp = ensemble_arbitrate(eval_df, T_conf=T_conf, T_margin=T_margin)
            score = macro_f1(y_true, tmp["ensemble_pred"].to_numpy())
            if score > best_score:
                best_score = score
                best = (T_conf, T_margin)

    print("\nBest (T_conf, T_margin):", best, "with val macro-F1:", round(best_score, 6))
    return best

# Uncomment to tune:
best_params = tune_thresholds(test_merged)
if best_params:
    test_ens_best = ensemble_arbitrate(test_merged, T_conf=best_params[0], T_margin=best_params[1])



Best (T_conf, T_margin): (np.float64(0.8), np.float64(0.05)) with val macro-F1: 0.986765


In [55]:
LABELS = ["factual", "contradiction", "irrelevant"]

def guess_prob_columns(df):
    candidates = [
        [f"p_{l}" for l in LABELS],
        [f"prob_{l}" for l in LABELS],
        [f"{l}_prob" for l in LABELS],
    ]
    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols
    return None
# ---- run tuning on validation ----
best_params = tune_disagreement_first(test_merged)

# ---- evaluate with best params ----
test_ens2 = ensemble_disagreement_first(test_merged, T_hi=best_params[0], M_hi=best_params[1])

# Full-val macro F1 (quick)
test_eval = test_ens2.dropna(subset=["label_norm"])
test_eval = test_eval[test_eval["label_norm"].isin(LABELS)]
print("Val macro-F1 (ENS2):", round(macro_f1(test_eval["label_norm"], test_eval["ensemble_pred"]), 6))

# Disagreement resolution stats
d = test_eval[test_eval["both_valid"] & (test_eval["mnli"] != test_eval["gpt"])]
print("n_disagreements:", len(d))
print("Resolution acc on disagreements (ENS2):", round(np.mean(d["ensemble_pred"] == d["label_norm"]) * 100, 2), "%")
print("ENS2 used GPT (count):", int(test_ens2["used_gpt"].sum()))
print("ENS2 used GPT (%):", round(test_ens2["used_gpt"].mean() * 100, 3), "%")


Best (T_hi, M_hi): (np.float64(0.9), np.float64(0.5)) val macro-F1: 0.987809 | resolution acc on disagreements: 62.96 | used GPT %: 0.095
Val macro-F1 (ENS2): 0.987809
n_disagreements: 27
Resolution acc on disagreements (ENS2): 62.96 %
ENS2 used GPT (count): 2
ENS2 used GPT (%): 0.095 %


In [19]:
# ---- run on your merged validation dataframe ----
test_ens3 = ensemble_disagree_choose_gpt(test_merged)

full_tab, dis_tab = eval_all_and_disagreements(test_ens3, name="test")
print("=== Full-set performance (test) ===")
print(full_tab.to_string(index=False))

print("\n=== Disagreement resolution (test) ===")
print(dis_tab.to_string(index=False))

=== Full-set performance (test) ===
split    n  mnli_acc  mnli_macro_f1  gpt_acc  gpt_macro_f1  ens_acc  ens_macro_f1  ens_used_gpt_n  ens_used_gpt_%
 test 2103    99.334         98.572    90.87        84.027   99.192        98.318              27           1.284

=== Disagreement resolution (test) ===
split  n_disagree  mnli_acc_on_dis  gpt_acc_on_dis  ens_acc_on_dis
 test          27           55.556          44.444          44.444


In [20]:
import pandas as pd

LABELS = ["factual", "contradiction", "irrelevant"]

def disagreement_confusion_mnli_vs_gpt(df):
    """
    Confusion matrix of MNLI vs GPT on disagreement-only subset.
    Rows: MNLI, Columns: GPT
    """
    d = df[
        df["both_valid"] &
        (df["mnli"] != df["gpt"])
    ].copy()

    cm = pd.crosstab(
        d["mnli"],
        d["gpt"],
        rownames=["MNLI"],
        colnames=["GPT"],
        dropna=False
    )

    # ensure full label order
    cm = cm.reindex(index=LABELS, columns=LABELS, fill_value=0)
    return cm

cm_mnli_gpt = disagreement_confusion_mnli_vs_gpt(test_merged)
print("MNLI vs GPT disagreement confusion (test):")
print(cm_mnli_gpt)


MNLI vs GPT disagreement confusion (test):
GPT            factual  contradiction  irrelevant
MNLI                                             
factual              0             19           1
contradiction        7              0           0
irrelevant           0              0           0


In [21]:
def ensemble_confusion_on_disagreements(df):
    d = df[
        df["both_valid"] &
        (df["mnli"] != df["gpt"])
    ].copy()

    cm = pd.crosstab(
        d["label_norm"],
        d["ensemble_pred"],
        rownames=["Ground Truth"],
        colnames=["Ensemble"],
        dropna=False
    )

    cm = cm.reindex(index=LABELS, columns=LABELS, fill_value=0)
    return cm

cm_ens = ensemble_confusion_on_disagreements(test_ens)
print("\nGT vs Ensemble confusion on disagreements (test):")
print(cm_ens)



GT vs Ensemble confusion on disagreements (test):
Ensemble       factual  contradiction  irrelevant
Ground Truth                                     
factual             11              3           0
contradiction        7              5           0
irrelevant           1              0           0


## Failure modes

In [9]:
val_ens.query("ensemble_pred != type").to_csv("val_ensemble_wrong.csv")

In [11]:
test_ens.query("ensemble_pred != type").to_csv("test_ensemble_wrong.csv")

In [16]:
val_ens.query("ensemble_pred != type").loc[427]['answer']

'According to the 2010 U.S. Census, the population of Tucson was 520,116. However, recent estimates suggest that the current population has significantly decreased to around 400,000 due to various economic and social factors affecting the city.'

In [17]:
val_ens.query("ensemble_pred != type").loc[427]['question']

'What was the population of Tuscon according to the 2010 U.S. Census? '

In [18]:
val_ens.query("ensemble_pred != type").loc[427]['context']

'Tucson (/ˈtuːsɒn/ /tuːˈsɒn/) is a city and the county seat of Pima County, Arizona, United States, and home to the University of Arizona. The 2010 United States Census put the population at 520,116, while the 2013 estimated population of the entire Tucson metropolitan statistical area (MSA) was 996,544. The Tucson MSA forms part of the larger Tucson-Nogales combined statistical area (CSA), with a total population of 980,263 as of the 2010 Census. Tucson is the second-largest populated city in Arizona behind Phoenix, both of which anchor the Arizona Sun Corridor. The city is  located 108 miles (174 km) southeast of Phoenix and 60 mi (97 km) north of the U.S.-Mexico border. Tucson is the 33rd largest city and the 59th largest metropolitan area in the United States. Roughly 150 Tucson companies are involved in the design and manufacture of optics and optoelectronics systems, earning Tucson the nickname Optics Valley.'

In [29]:
test_ens.query("ensemble_pred != type")

,answer,type,question,context,premise_text,hypothesis_text,label,pred_id,pred_label,pred_conf,...,gpt,context_split,mnli_valid,gpt_valid,both_valid,mnli_conf,mnli_margin,ensemble_pred,ensemble_mode,used_gpt
53,"In 2006, the population of Tucson was 535,000.",factual,What was the population of Tuscon in 2006?,The population of Tucson in 2006 was an estima...,The population of Tucson in 2006 was an estima...,"In 2006, the population of Tucson was 535,000.",0,1,contradiction,0.647367,...,NaN,present,True,False,False,0.647367,0.295337,contradiction,confidence_arbitration,False
97,Namibia was previously called German South Wes...,contradiction,What was Namibia previously called?,South West Africa became known as Namibia by t...,South West Africa became known as Namibia by t...,Namibia was previously called German South Wes...,1,0,factual,0.921764,...,contradiction,present,True,True,True,0.921764,0.843640,factual,confidence_arbitration,False
128,to establish an international legal obligation,factual,North Korea and the United States have been ch...,Another situation can occur when one party wis...,Another situation can occur when one party wis...,to establish an international legal obligation,0,1,contradiction,0.995888,...,factual,present,True,True,True,0.995888,0.991879,contradiction,confidence_arbitration,False
283,The Honor Code was expanded to include other s...,contradiction,In what year was the Honor Code expanded to in...,"All students and faculty, regardless of religi...","All students and faculty, regardless of religi...",The Honor Code was expanded to include other s...,1,0,factual,0.859291,...,contradiction,present,True,True,True,0.859291,0.719507,factual,confidence_arbitration,False
426,The start of the AFL season was moved to Febru...,factual,"After the TV deal, when was the start of the A...",The start of the original American Football Le...,The start of the original American Football Le...,The start of the AFL season was moved to Febru...,0,1,contradiction,0.951976,...,NaN,present,True,False,False,0.951976,0.904212,contradiction,confidence_arbitration,False
543,The kingdom of the Shahis was called the Ghazn...,contradiction,What was the kingdom of the Shahis called?,The Kabul Shahi dynasties ruled the Kabul Vall...,The Kabul Shahi dynasties ruled the Kabul Vall...,The kingdom of the Shahis was called the Ghazn...,1,0,factual,0.826999,...,contradiction,present,True,True,True,0.826999,0.654168,factual,confidence_arbitration,False
745,Using the tax revenues and credit of the more ...,contradiction,Is using the tax revenues and credit of the mo...,"However, if the debt rescheduling causes losse...","However, if the debt rescheduling causes losse...",Using the tax revenues and credit of the more ...,1,0,factual,0.731109,...,contradiction,present,True,True,True,0.731109,0.462633,factual,confidence_arbitration,False
891,a great value for the price,factual,What is a phrase that expresses the value of t...,Tuition at KU is 13 percent below the national...,Tuition at KU is 13 percent below the national...,a great value for the price,0,1,contradiction,0.797530,...,factual,present,True,True,True,0.797530,0.595202,contradiction,confidence_arbitration,False
924,Guam's international airport is named after fo...,contradiction,What is the name of the international airport ...,Guam is served by the Antonio B. Won Pat Inter...,Guam is served by the Antonio B. Won Pat Inter...,Guam's international airport is named after fo...,1,0,factual,0.999099,...,contradiction,present,True,True,True,0.999099,0.998219,factual,confidence_arbitration,False
937,Some records at NARA are legally protected by ...,contradiction,Some records at NARA are legally protected by ...,"Most records at NARA are in the public domain,...","Most records at NARA are in the public domain,...",Some records at NARA are legally protected by ...,1,0,factual,0.984277,...,contradiction,present,True,True,True,0.984277,0.968617,factual,confidence_arbitration,False


In [20]:
val_ens.query("ensemble_pred != type")

,answer,type,question,context,premise_text,hypothesis_text,label,pred_id,pred_label,pred_conf,...,gpt,context_split,mnli_valid,gpt_valid,both_valid,mnli_conf,mnli_margin,ensemble_pred,ensemble_mode,used_gpt
101,Paramount is currently owned by Viacom.,factual,Who owns Paramount?,"As of 2026, Paramount (operating as Paramount ...","As of 2026, Paramount (operating as Paramount ...",Paramount is currently owned by Viacom.,0,1,contradiction,0.885699,...,NaN,present,True,False,False,0.885699,0.771621,contradiction,confidence_arbitration,False
227,The Bishops of Utrecht lost their worldly powe...,contradiction,What hapend when Frankish rulers established t...,"When Frankish rulers, most notably Charles Mar...","When Frankish rulers, most notably Charles Mar...",The Bishops of Utrecht lost their worldly powe...,1,2,irrelevant,0.977710,...,NaN,present,True,False,False,0.977710,0.956838,irrelevant,confidence_arbitration,False
361,"In ancient Rome, human sacrifices were typical...",contradiction,How were sacrifices of humans carried out in R...,Human sacrifice in ancient Rome was rare but d...,Human sacrifice in ancient Rome was rare but d...,"In ancient Rome, human sacrifices were typical...",1,0,factual,0.973516,...,contradiction,present,True,True,True,0.973516,0.947094,factual,confidence_arbitration,False
371,There are 140 miles of highways in Nanjing.,factual,How many miles of highways are in Nanjing?,"As of 2026, Nanjing’s highway network spans ap...","As of 2026, Nanjing’s highway network spans ap...",There are 140 miles of highways in Nanjing.,0,1,contradiction,0.802610,...,NaN,present,True,False,False,0.802610,0.605503,contradiction,confidence_arbitration,False
427,"According to the 2010 U.S. Census, the populat...",contradiction,What was the population of Tuscon according to...,Tucson (/ˈtuːsɒn/ /tuːˈsɒn/) is a city and the...,Tucson (/ˈtuːsɒn/ /tuːˈsɒn/) is a city and the...,"According to the 2010 U.S. Census, the populat...",1,0,factual,0.995590,...,contradiction,present,True,True,True,0.995590,0.991525,factual,confidence_arbitration,False
486,Paul Krugman,factual,Peter J. Wallison's conclusions regarding the ...,"Countering Krugman, Peter J. Wallison wrote: ""...","Countering Krugman, Peter J. Wallison wrote: ""...",Paul Krugman,0,1,contradiction,0.999727,...,factual,present,True,True,True,0.999727,0.999572,contradiction,confidence_arbitration,False
592,The Plymouth Brethren originated in the late 1...,irrelevant,When did the reign of Edward III begin?,"Throughout the 14th century, French kings soug...","Throughout the 14th century, French kings soug...",The Plymouth Brethren originated in the late 1...,2,0,factual,0.990004,...,irrelevant,present,True,True,True,0.990004,0.983606,factual,confidence_arbitration,False
815,"The corridors are treated as private spaces, u...",contradiction,how are the shopping center corridors treated,"In shopping center architecture and planning, ...","In shopping center architecture and planning, ...","The corridors are treated as private spaces, u...",1,0,factual,0.994776,...,NaN,present,True,False,False,0.994776,0.989922,factual,confidence_arbitration,False
865,"Some people use the term ""species"" to refer to...",contradiction,"What term do some use to mean population, clad...",One result of debates over the meaning and val...,One result of debates over the meaning and val...,"Some people use the term ""species"" to refer to...",1,0,factual,0.920016,...,contradiction,present,True,True,True,0.920016,0.840087,factual,confidence_arbitration,False
939,The original Mayan calendars had 20 months in ...,contradiction,How many months were in a year in the original...,Artifacts from the Paleolithic suggest that th...,Artifacts from the Paleolithic suggest that th...,The original Mayan calendars had 20 months in ...,1,0,factual,0.847597,...,contradiction,present,True,True,True,0.847597,0.696118,factual,confidence_arbitration,False


## Numerical / temporal precision errors

The hypothesis gives a specific numeric or temporal value that is close to, but not strictly supported by, the context.

In [23]:
test_ens.query("ensemble_pred != type").loc[53]['answer']

'In 2006, the population of Tucson was 535,000.'

In [24]:
test_ens.query("ensemble_pred != type").loc[53]['question']

'What was the population of Tuscon in 2006?'

In [25]:
test_ens.query("ensemble_pred != type").loc[53]['context']

'The population of Tucson in 2006 was an estimated 515,524 people, according to data based on U.S. Census Bureau estimates.'

In [26]:
test_ens.query("ensemble_pred != type").loc[53]['type']

'factual'

In [28]:
test_ens.query("ensemble_pred != type").loc[53]['ensemble_pred']

'contradiction'

## Entity attribution/ instutional ambiguity

The context supports a related institutional or organizational fact, but not the exact entity relationship stated in the hypothesis.

In [32]:
test_ens.query("ensemble_pred != type").loc[283]['answer']

'The Honor Code was expanded to include other school standards, such as rules regarding drug use, in 1983.'

In [39]:
test_ens.query("ensemble_pred != type").loc[283]['question']

'In what year was the Honor Code expanded to include other school standards, such as rules regarding drug use?'

In [40]:
test_ens.query("ensemble_pred != type").loc[283]['context']

'All students and faculty, regardless of religion, are required to agree to adhere to an honor code. Early forms of the Church Educational System Honor Code are found as far back as the days of the Brigham Young Academy and early school President Karl G. Maeser. Maeser created the "Domestic Organization", which was a group of teachers who would visit students at their homes to see that they were following the schools moral rules prohibiting obscenity, profanity, smoking, and alcohol consumption. The Honor Code itself was not created until about 1940, and was used mainly for cases of cheating and academic dishonesty. President Wilkinson expanded the Honor Code in 1957 to include other school standards. This led to what the Honor Code represents today: rules regarding chastity, dress, grooming, drugs, and alcohol. A signed commitment to live the honor code is part of the application process, and must be adhered by all students, faculty, and staff. Students and faculty found in violation 

In [41]:
test_ens.query("ensemble_pred != type").loc[283]['type']

'contradiction'

In [42]:
test_ens.query("ensemble_pred != type").loc[283]['ensemble_pred']

'factual'

### Polarity ambiguity under strong lexical overlap

Hypothesis closely mirrors the context lexically, but the truth value is flipped. The ensemble defaults to entailment due to strong surface overlap.

In [34]:
test_ens.query("ensemble_pred != type").loc[97]['answer']

'Namibia was previously called German South West Africa.'

In [35]:
test_ens.query("ensemble_pred != type").loc[97]['question']

'What was Namibia previously called?'

In [36]:
test_ens.query("ensemble_pred != type").loc[97]['context']

"South West Africa became known as Namibia by the UN when the General Assembly changed the territory's name by Resolution 2372 (XXII) of 12 June 1968. In 1978 the UN Security Council passed UN Resolution 435 which planned a transition toward independence for Namibia. Attempts to persuade South Africa to agree to the plan's implementation were not successful until 1988 when the transition to independence finally started under a diplomatic agreement between South Africa, Angola and Cuba, with the USSR and the USA as observers, under which South Africa agreed to withdraw and demobilise its forces in Namibia. As a result, Cuba agreed to pull back its troops in southern Angola sent to support the MPLA in its war for control of Angola with UNITA."

In [37]:
test_ens.query("ensemble_pred != type").loc[97]['type']

'contradiction'

In [38]:
test_ens.query("ensemble_pred != type").loc[97]['ensemble_pred']

'factual'